<h1>Math</h1>

omitted

<h1>Math</h1>

omitted

<h1>Math</h1>

omitted

<h1>Math</h1>

omitted

<h1>Math</h1>

omitted

<h1>Math</h1>

omitted

<h1>code</h1>

In [342]:
import numpy as np
import pandas as pd

In [343]:
data = pd.read_csv('datasets/site_data.csv',header = None)
data.head()

,0,1
0,-1,8
1,4,8
2,-1,2
3,1,B
4,-1,5


In [352]:
# give B and C numeric ids , B --> 10 , C --> 11
data = data.replace({'B':10},regex=True)
data = data.replace({'C':11},regex=True)

In [353]:
# see the first this means 8 is a start page
# see the third column , so user visited page 1 then bounced
X = data.to_numpy().astype('int32')
# get initial pages to calcualte pi
initials = X[X[:,0]==-1]
# get transitions
transitions = X[X[:,0]!=-1]
# we have 12 states : 10 pages + B + C
M = 12

In [354]:
# lets fill in pi , we will use add one smoothing
pi = np.ones(M) # this can be M-2 since we cant start in B or C
# first we count
for _,page in initials:
    pi[page]+=1
#next we get probabilities
pi = pi/(pi.sum())
# we can also convert these to log probabilities
#pi = np.log(pi)

In [355]:
# next lets fill A
A = np.ones((M,M))
# first are counts
for from_,to in transitions:
    A[from_][to] += 1
# then lets change these to probabilities
A = A/A.sum(axis=1,keepdims=True)
# log probabilities
#A = np.log(A)

In [275]:
# now lets print intiial state distribution
print('initial state distribution: ')
print(pi) # we cant start at B or C , but they have low probs ince we use add one smoothing
print('page with highest probability to start at : ',pi.argmax())

initial state distribution: 
[1.02974483e-01 1.03477779e-01 9.50727264e-02 9.51230560e-02
 1.02420857e-01 9.77905280e-02 9.79918466e-02 9.97030550e-02
 1.01514923e-01 1.03830087e-01 5.03296593e-05 5.03296593e-05]
page with highest probability to start at :  9


In [276]:
# now lets get bounce rate for each page
print('bounce rate for each page')
print(A[:,10]) # no bounce rates for B or C , still they get low probs due to add one smoothinh
print('page with highest bounce rate : ',A[:,10].argmax())

bounce rate for each page
[0.12790123 0.12587671 0.12643106 0.1273679  0.12551261 0.12363405
 0.12075984 0.12365591 0.12523295 0.1316888  0.08333333 0.08333333]
page with highest bounce rate :  9


<h1>Math</h1>

omitted

<h1>code</h1>

In [85]:
import numpy as np
import string

In [86]:
def tokenise(sent):
    # remove punctuation
    sent = sent.rstrip()
    sent = sent.translate(str.maketrans('','',string.punctuation))
    tokens = [w for w in sent.lower().split()]
    return tokens

In [87]:
sentences = []
# read poems
for line in open('datasets/robert_frost.txt'):
    # add <END>token and append sentence
    tokens = tokenise(line)
    if len(tokens) > 0:
        sentences.append( tokens + ['<END>'])


In [88]:
# initial character distribution , p(w1) , where w1 is the first word of a sentence
pi = {}
# second character distribution , p( w2 | w1) # where w2 is the second word of a sentence
second = {}
# state transition , our 2nd-order model p( w(t) | w(t-2),w(t-1) )
A = {}

In [89]:
# first lets fill in pi
for sentence in sentences:
    pi[sentence[0]] = pi.get(sentence[0],0) + 1
        
# now change to probabilities
pi_sum = sum(pi.values())

for k,v in pi.items():
    pi[k] = v/pi_sum

In [90]:
# next lets fill in second

for sentence in sentences:
    w1,w2 = sentence[0],sentence[1]
    if w1 not in second.keys():
        second[w1] = {}
    second[w1][w2] = second[w1].get(w2,0)+1
        
# now change to probability

for k,word2count in second.items():
    sum_counts = sum(word2count.values())
    
    for word,count in word2count.items():
        second[k][word] = count/sum_counts

In [91]:
# now A , this is the same as second with a sequence of three words instead of three

for sentence in sentences:
    for i in range(len(sentence)-2):
        w1,w2,w3 = sentence[i],sentence[i+1],sentence[i+2]
        
        if (w1,w2) not in A.keys():
            A[(w1,w2)] = {}
        A[(w1,w2)][w3] = A[(w1,w2)].get(w3,0)+1
        
# now change to probability

for k,word2count in A.items():
    sum_counts = sum(word2count.values())
    
    for word,count in word2count.items():
        A[k][word] = count/sum_counts

In [92]:
# now lets generate a line

# we will be sampling words as we will see shortly
# we will stop whenever we sample the token <END>
# or whenever we reach the max len of a sentence
max_len = max(len(sentence) for sentence in sentences) - 1 # since these have <END> token added

def sample_word(word2prop):
    i = np.random.choice(len(word2prop),p = list(word2prop.values()))
    w = list(word2prop.keys())[i]
    return w

def sample_line():
    line = []
    # first we use pi probabilities to sample the first word
    w1 = sample_word(pi)
    line.append(w1)
    # next we want to use second to sample the second word given first
    w2 = sample_word(second[w1])
    
    if w2 == '<END>':
        return ' '.join(line) 
    line.append(w2)
    
    for _ in range(max_len-2):
        # sample a word given previous 2 words
        w1,w2 = line[-2],line[-1]
        # now lets sample w3 from A
        w3 = sample_word(A[(w1,w2)])
        if w3 == '<END>':
            return ' '.join(line)
        line.append(w3)

    return ' '.join(line) 

In [93]:
def sample_stanza(n=4):
    for _ in range(n):
        print(sample_line())

In [108]:
sample_stanza()

looked up toward heaven and there a tent in grove and
two winds would meet
thats no way for a home for me in the mountain
i shall be telling this with a hornyhanded kindness


In [114]:
def generate():
    while True:
        sample_stanza()
        print('-----------------------------------------------')
        r = input('one more [Y/n] ')
        print('-----------------------------------------------')
        if r.lower() == 'n':
            break

In [113]:
generate()

but i dont trust your eyes youve said enough
old davis owned a solid mica mountain
life long in that grave word
hes got hay down thats been there for hawks since chickentime
-----------------------------------------------
one more [Y/n] Y
-----------------------------------------------
like a bird silent in flight
yet does perhaps for all three
what bones the cellar to the cellar
to find yourself
-----------------------------------------------
one more [Y/n] n
-----------------------------------------------


<h1>Math</h1>

omitted

<h1>Code</h1>

In [479]:
# lets take a quick example on the PF criterion
M = 20
# create a random matrix , sample from unigorm distribution [0,1]
A = np.random.rand(M,M)
# just make sure A is strictly positive
A = A + 1e-8
# now normalise so each row sum up to 1
A = A/A.sum(axis=1,keepdims=True)
# now A satisfies the PF criterion

In [597]:
# 1 : get limiting distribution iteratively , we know this converges as time approaches infinity
v1 = np.random.rand(M)
# normalise so its a valid probability distribution
v1 = v1/v1.sum()

for i in range(1000):
    # or v1@A if v is a 1xM vector these are the same 
    # but this for looks more like the eigenvalue problem
    # explais why we will feed in A.T not A
    v1 = A.T@v1 
    # make sure it remains a valid probability distribution
v1 = v1/v1.sum()

In [598]:
# 2 : get statinoary distribution using eigens as we have shown before
evals,evecs = np.linalg.eig(A.T)

# get index where eigen value = 1
i = np.argmax(np.isclose(evals,1))
# get corresponding eigen value
v2 = evecs[:,i]
v2 = np.real(v2)
# normalise
v2 = v2/v2.sum()

In [599]:
print(np.isclose(A.T@v2,v2).all())
print(np.isclose(A.T@v1,v1).all())
print(np.isclose(v1,v2).all())

True
True
True


In [600]:
print(v1)
print('---------------')
print(v2)

[0.03886189 0.05215062 0.04046588 0.05911771 0.04666012 0.04141172
 0.05218089 0.05025517 0.05706079 0.04281098 0.05228438 0.05308884
 0.04950908 0.05474549 0.05838592 0.04162408 0.05790279 0.04501665
 0.05539791 0.05106911]
---------------
[0.03886189 0.05215062 0.04046588 0.05911771 0.04666012 0.04141172
 0.05218089 0.05025517 0.05706079 0.04281098 0.05228438 0.05308884
 0.04950908 0.05474549 0.05838592 0.04162408 0.05790279 0.04501665
 0.05539791 0.05106911]


In [574]:
# so we get our unique stationary distribution which is also the limiting distribution